## 1.Brute Force
The brute-force method enumerates all permutations, and its computational complexity explodes factorially.

Essentially, the brute-force approach for Flow Shop Scheduling (FSP) works as follows:

enumerate every processing sequence of jobs → calculate the makespan for each sequence → search for the optimal solution.

The total number of processing sequences for n jobs equals n! (n factorial).

The test instances in the flowshop1.txt dataset fall into three categories:
- car1–car8: 8 jobs → can just be solved by the brute-force method;
- hel1–hel2: a slightly larger number of jobs → the brute-force method runs very slowly;
- rec01–rec42: 20 to 100 jobs → cannot be computed by the brute-force method within a reasonable time.
Therefore, brute-force enumeration is temporarily unsuitable for conducting this comparison.


## 2.Johnsons Algorithm
The classic Johnson algorithm guarantees the optimal sequence only for two-machine flow shop scheduling. Its mathematical optimality proof cannot be extended directly to more machines.For three-machine cases, the virtual machine method combines the processing time of adjacent real machines into two virtual machines to apply the two-machine Johnson rule approximately. However, this approach requires strict restrictive conditions and cannot ensure optimality.

## 2.1 Construction rule of the virtual machine method:
Let the actual processing time of each job be defined as follows:
- t1: processing time on Machine 1
- t2: processing time on Machine 2
- t3: processing time on Machine 3
Two combined virtual machines are constructed:
- Virtual Machine A: V1 = t1 + t2 
- Virtual Machine B: V2 = t2 + t3 

After construction, the standard two‑machine Johnson’s rule is applied to the pair (V1,V2) to obtain the optimal processing sequence.

## 2.2 Application Prerequisite (not satisfied in most practical scenarios)
The virtual machine method yields near‑optimal results only if one of the following constraints holds:
- max(All jobs’ t1 ) ≤ min(All jobs’ t2 ) or max(All jobs’ t3 ) ≤ min(All jobs’ t2)

## 2.3 Physical interpretation
The intermediate Machine 2 will never suffer from starvation or idle blocking.
If these conditions are violated, the obtained sequence is merely an approximate solution rather than a global optimum, and it will produce substantial computational errors.
Since all instances in flowshop1.txt contain 5 or more machines, the Johnson algorithm and its virtual-machine extension are not applicable.



## 3 NEH Algorithm
The NEH algorithm is a heuristic algorithm for permutation flow shop scheduling (PFSP). Its main goal is to find a good initial processing sequence quickly.

The NEH algorithm has three simple steps. 

First, it calculates the total processing time of each job. The total time is the sum of the job’s processing time on all machines. 

Second, it sorts all jobs in descending order of their total processing time. 

Third, it builds the optimal sequence step by step. It starts with the first job, then inserts each subsequent job into all possible positions. It keeps the position that gives the shortest makespan (total completion time).

The NEH algorithm is fast. It can handle large-scale instances (such as 100 jobs × 10 machines) in seconds. It provides a "good" initial solution, which is close to the optimal solution.


In [2]:
import re
import collections
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from ortools.sat.python import cp_model

In [4]:
def parse_flowshop1(file_path: str = "flowshop1.txt"):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        raw_lines = [line.strip() for line in f.readlines()]

    instances = []
    idx = 0
    total = len(raw_lines)

    # Skip all the lines at the beginning of the file until find the first 'instance' line
    while idx < total and not raw_lines[idx].startswith("instance"):
        idx += 1

    # Iteratively parse each instance (marked by 'instance')
    while idx < total:
        if raw_lines[idx].startswith("instance"):
            # Read instance name
            instance_name = raw_lines[idx].replace("instance", "").strip()
            idx += 1

            # skip ++++
            while idx < total and raw_lines[idx].startswith("+"):
                idx += 1

            # Find line with exactly 2 numbers: jobs count and machines count
            jobs_count, machines_count = 0, 0
            while idx < total:
                num_list = re.findall(r'\d+', raw_lines[idx])
                if len(num_list) == 2:
                    jobs_count = int(num_list[0])
                    machines_count = int(num_list[1])
                    idx += 1
                    break
                idx += 1

            # Read job processing data (strictly read jobs_count valid lines)
            jobs_data = []
            valid_lines = 0
            while valid_lines < jobs_count and idx < total:
                line = raw_lines[idx]
                nums = list(map(int, re.findall(r'\d+', line)))
                idx += 1

                if len(nums) >= 2 and len(nums) % 2 == 0:
                    process_times = [nums[i+1] for i in range(0, len(nums), 2)][:machines_count]
                    if len(process_times) == machines_count:
                        jobs_data.append(process_times)
                        valid_lines += 1

            # Store valid instance
            if len(jobs_data) == jobs_count:
                instances.append({
                    "name": instance_name,
                    "jobs_count": jobs_count,
                    "machines_count": machines_count,
                    "jobs_data": jobs_data  
                })
        else:
            idx += 1
    return instances

In [5]:
fsp_instances = parse_flowshop1("flowshop1.txt")
print(f"Successfully loaded {len(fsp_instances)} FSP test instances")

if fsp_instances:
    ins = fsp_instances[0]
    print(f"\nFirst Instance Info:")
    print(f"Name: {ins['name']}")
    print(f"Jobs: {ins['jobs_count']} | Machines: {ins['machines_count']}")

Successfully loaded 31 FSP test instances

First Instance Info:
Name: car1
Jobs: 11 | Machines: 5


In [6]:
from KamilGos.neh import *

In [7]:
# Run all instance
for idx, instance in enumerate(fsp_instances):
    print(f"\n[{idx+1}/{len(fsp_instances)}] Instance: {instance['name']}")
    print(f"Jobs: {instance['jobs_count']} | Machines: {instance['machines_count']}")
    
    tasks_val = instance["jobs_count"]
    machines_val = instance["machines_count"]
    tasks = instance["jobs_data"]

    # result table
    table = PrettyTable()
    table.field_names = ["Algorithm", "Makespan (Cmax)", "Execution Time (s)"]

    # Classic NEH
    start = time.perf_counter()
    seq, cmax_classic = neh(tasks, machines_val, tasks_val)
    table.add_row(["Classic NEH", cmax_classic, round(time.perf_counter()-start, 4)])

    # Accelerated NEH
    start = time.perf_counter()
    seq, cmax_acc = neh_acc(tasks, machines_val, tasks_val)
    table.add_row(["Accelerated NEH", cmax_acc, round(time.perf_counter()-start, 4)])

    # Modified NEH
    start = time.perf_counter()
    seq, cmax_wm = neh_wm(tasks, machines_val, tasks_val)
    table.add_row(["Modified NEH", cmax_wm, round(time.perf_counter()-start, 4)])

    print(table)

print("\n" + "=" * 80)
print("ALL INSTANCES EXECUTED SUCCESSFULLY!")
print("=" * 80)


[1/31] Instance: car1
Jobs: 11 | Machines: 5
+-----------------+-----------------+--------------------+
|    Algorithm    | Makespan (Cmax) | Execution Time (s) |
+-----------------+-----------------+--------------------+
|   Classic NEH   |       7038      |       0.0018       |
| Accelerated NEH |       7038      |       0.0009       |
|   Modified NEH  |       7038      |       0.0056       |
+-----------------+-----------------+--------------------+

[2/31] Instance: car2
Jobs: 13 | Machines: 4
+-----------------+-----------------+--------------------+
|    Algorithm    | Makespan (Cmax) | Execution Time (s) |
+-----------------+-----------------+--------------------+
|   Classic NEH   |       7376      |       0.0014       |
| Accelerated NEH |       7376      |       0.0009       |
|   Modified NEH  |       7376      |       0.0021       |
+-----------------+-----------------+--------------------+

[3/31] Instance: car3
Jobs: 12 | Machines: 5
+-----------------+-----------------

## 4.Simulated Annealing
## 4.1 Simulated Annealing Algorithm 
The simulated annealing (SA) algorithm is an optimization algorithm. It imitates the annealing process of metals.

When the temperature is high, the algorithm explores more possible solutions. It may accept worse solutions to avoid getting stuck in local optimal solutions. When the temperature cools down gradually, it only keeps better solutions. This helps the algorithm converge to the global optimal solution.
The SA algorithm has four key steps. 

First, it gets an initial solution (usually from the NEH algorithm). 

Second, it sets an initial temperature and a cooling rate. 

Third, it generates a new solution by making small changes to the current solution. 

Fourth, it decides whether to accept the new solution based on the Metropolis criterion. 

It repeats these steps until the temperature is low enough.
## 4.2 NEH + Simulated Annealing Algorithm
It combine the NEH algorithm with the SA algorithm to get better results. This combination takes advantage of both algorithms.
The NEH algorithm provides a good initial solution quickly. This avoids the SA algorithm’s blind search at the beginning. Then the SA algorithm optimizes this initial solution. It explores more possible sequences and finds a better solution than the NEH algorithm alone.

This combination is very suitable for large-scale PFSP instances, such as the hel1 instance (100 jobs × 10 machines). It runs fast (within 2 seconds) and gives high-quality results. It is much more efficient than exact solvers like OR-Tools for large instances.

In [8]:
from KamilGos.simulated_annealing import *

In [12]:
# 模拟退火参数
T_START = 5000
T_END = 10

# 遍历所有用例（包括 hel1 100×10）
for i, instance in enumerate(fsp_instances):
    print("\n" + "="*70)
    print(f"[{i+1}/{len(fsp_instances)}] Solving Instance: {instance['name']}")
    print(f"Jobs: {instance['jobs_count']} | Machines: {instance['machines_count']}")

    n = instance["jobs_count"]
    m = instance["machines_count"]
    tasks = instance["jobs_data"]  # 直接传入SA+NEH代码

    # 运行算法（hel1 1秒内出结果）
    start = time.time()
    best_seq, best_cmax = sim_ann1(n, tasks, m, T_START, T_END)
    solve_time = round(time.time() - start, 4)

    print(f" Solve Time: {solve_time}s")
    print(f" Best Makespan: {best_cmax}")
    print(f" Optimal Sequence: {best_seq[:10]}...")  # 长序列只打印前10个
    print("="*70)


[1/31] Solving Instance: car1
Jobs: 11 | Machines: 5
 Solve Time: 0.0086s
 Best Makespan: 7038
 Optimal Sequence: [7, 2, 0, 10, 8, 3, 1, 4, 6, 5]...

[2/31] Solving Instance: car2
Jobs: 13 | Machines: 4
 Solve Time: 0.0075s
 Best Makespan: 7376
 Optimal Sequence: [6, 10, 12, 3, 2, 7, 1, 5, 8, 11]...

[3/31] Solving Instance: car3
Jobs: 12 | Machines: 5
 Solve Time: 0.0097s
 Best Makespan: 7399
 Optimal Sequence: [10, 5, 11, 9, 4, 8, 2, 3, 1, 7]...

[4/31] Solving Instance: car4
Jobs: 14 | Machines: 4
 Solve Time: 0.0083s
 Best Makespan: 8003
 Optimal Sequence: [3, 13, 6, 12, 5, 11, 8, 7, 9, 0]...

[5/31] Solving Instance: car5
Jobs: 10 | Machines: 6
 Solve Time: 0.0084s
 Best Makespan: 7832
 Optimal Sequence: [2, 3, 4, 0, 9, 8, 5, 1, 6, 7]...

[6/31] Solving Instance: car6
Jobs: 8 | Machines: 9
 Solve Time: 0.0088s
 Best Makespan: 8742
 Optimal Sequence: [7, 4, 6, 3, 5, 2, 0, 1]...

[7/31] Solving Instance: car7
Jobs: 7 | Machines: 7
 Solve Time: 0.008s
 Best Makespan: 6590
 Optimal S